# BARE Pipeline - Main Execution Notebook

This notebook serves as the main execution interface for the BARE (Boundary-Aware with Resolution Enhancement) experimental codebase.

## Pipeline Overview

1. Setup & Configuration
2. Dataset Loading & Exploration
3. Model Creation
4. Training Pipeline
5. Evaluation & Metrics
6. Prediction on New Images
7. Advanced Visualization & Analysis

In [ ]:
# Add project root to system path (flat structure - all modules in same directory)
import sys
import os

# Get the directory containing this notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('.'))

# In flat structure, all modules are in the same directory as the notebook
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print(f"Project Path: {NOTEBOOK_DIR}")

In [ ]:
import torch
import logging
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import warnings
import json
import time
import glob

warnings.filterwarnings("ignore", category=UserWarning, module='PIL')

# Project-specific imports
from config import Config, adjust_config_for_architecture
from dataset import load_and_shuffle_dataset, create_dataloaders
from model import create_model
from pipeline import run_training_pipeline, evaluate_model, run_prediction_pipeline
from metrics import calculate_metrics
from utils import get_logger, set_seed
from checkpoint import load_model_for_evaluation

# Visualization imports
from visualization import (
    plot_confusion_matrix, plot_error_analysis_map, visualize_segmentation,
    plot_prediction_confidence, plot_enhanced_segmentation_analysis
)
from inspect_dataset import examine_raw_annotations, inspect_dataset_samples

%matplotlib inline

In [ ]:
# Setup logger and configuration
logger = get_logger()
config = Config()
config = adjust_config_for_architecture(config)

logger.info(f"Configuration loaded for '{config['architecture']}' architecture.")
logger.info(f"Output directory: {config['output_dir']}")
logger.info(f"Using seed: {config['seed']}")

In [ ]:
# Check device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Available CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Setup & Configuration

In [ ]:
# Save configuration
config_dict = config.to_dict() if hasattr(config, 'to_dict') else dict(config)
config_save_path = os.path.join(config['output_dir'], 'config.json')
os.makedirs(config['output_dir'], exist_ok=True)

with open(config_save_path, 'w') as f:
    json.dump(config_dict, f, indent=4)
logger.info(f"Configuration saved to {config_save_path}")



## 2. Dataset Loading & Exploration

In [ ]:
# Load and shuffle dataset
logger.info(f"Loading dataset: {config['dataset_name']}")
dataset_dict = load_and_shuffle_dataset(
    dataset_name=config['dataset_name'], 
    seed=config['seed']
)

# Architecture-specific target size
target_size = None
if config['architecture'] == 'setr':
    logger.info(f"SETR: Native {config['setr_input_size']}px processing.")

# Create dataloaders
train_dataloader, eval_dataloader, id2label, label2id = create_dataloaders(
    dataset_dict=dataset_dict,
    image_processor=None,
    config=config, 
    train_batch_size=config['train_batch_size'],
    eval_batch_size=config['eval_batch_size'],
    num_workers=config['num_workers'],
    validation_split=config['validation_split'],
    seed=config['seed'],
    target_size=target_size
)

config['id2label'] = id2label
config['label2id'] = label2id
logger.info(f"Train batches: {len(train_dataloader)}, Eval batches: {len(eval_dataloader)}")

In [ ]:
# Display sample
batch = next(iter(train_dataloader))
img_tensor = batch['pixel_values'][0]
lbl_tensor = batch['labels'][0]

img_display = img_tensor.permute(1, 2, 0).cpu().numpy()
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
img_display = np.clip(std * img_display + mean, 0, 1)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img_display)
ax[0].set_title('Sample Image')
ax[0].axis('off')
ax[1].imshow(lbl_tensor.cpu().numpy(), cmap='gray')
ax[1].set_title('Sample Mask')
ax[1].axis('off')
plt.show()

## 3. Model Creation

In [ ]:
num_training_steps = len(train_dataloader) * config['num_epochs'] // config['gradient_accumulation_steps']

model, optimizer, scheduler = create_model(
    config=config, 
    num_training_steps=num_training_steps,
    logger=logger
)

# Model summary — use architecture-appropriate input size
from torchinfo import summary
arch = config['architecture']
if arch == 'setr':
    input_size_h = config['setr_input_size']
elif arch == 'swin_unet':
    # Swin-UNet resizes internally to native resolution (e.g. 224)
    import re
    _match = re.search(r'_(\d+)$', config.get('swin_backbone', ''))
    input_size_h = int(_match.group(1)) if _match else 224
elif arch == 'mask2former':
    input_size_h = config.get('augmentation', {}).get('random_crop_size', 512)
else:
    input_size_h = config['augmentation']['random_crop_size']

try:
    summary(model, input_size=(config['train_batch_size'], 3, input_size_h, input_size_h))
except Exception as e:
    logger.warning(f"Model summary failed: {e}")

logger.info("Model, optimizer, and scheduler created.")

## 4. Training Pipeline

In [ ]:
logger.info("Starting training pipeline...")
training_start_time = time.time()

training_results = run_training_pipeline(
    config=config,
    logger=logger,
    is_notebook=True
)

training_duration = time.time() - training_start_time
print(f"\nTraining finished in {training_duration / 60:.2f} minutes.")
logger.info(f"Final model saved to: {training_results['model_dir']}")
logger.info(f"Final metrics: {training_results['metrics']}")

## 5. Evaluation & Metrics

In [ ]:
best_model_path = training_results.get('best_model_dir') or training_results['model_dir']
logger.info(f"Evaluating best model from: {best_model_path}")

model_eval, image_processor_eval = load_model_for_evaluation(
    model_path=best_model_path,
    config=config,
    device=device,
    logger=logger
)

_, eval_dataloader_reeval, _, _ = create_dataloaders(
    dataset_dict=dataset_dict,
    image_processor=image_processor_eval,
    config=config,
    train_batch_size=config['train_batch_size'],
    eval_batch_size=config['eval_batch_size'],
    num_workers=config['num_workers'],
    validation_split=config['validation_split'],
    seed=config['seed']
)

metrics = evaluate_model(
    model=model_eval,
    eval_dataloader=eval_dataloader_reeval,
    device=device,
    output_dir=config['output_dir'],
    id2label=config['id2label'],
    analyze_errors=True,
    logger=logger,
    is_notebook=True
)

logger.info(f"Evaluation metrics: {metrics}")

## 6. Prediction on New Images

In [ ]:
image_path = "/home/awilaiwo/Downloads/test_image.tif"  # Update with your image path

if os.path.exists(image_path):
    prediction_output_dir = os.path.join(config['output_dir'], 'predictions')
    
    prediction_results = run_prediction_pipeline(
        config=config,
        image_paths=image_path,
        model_path=training_results['model_dir'],
        output_dir=prediction_output_dir,
        visualize=True,
        show_confidence=True,
        logger=logger,
        is_notebook=True
    )
    
    if prediction_results['visualizations']:
        plt.figure(figsize=(12, 6))
        plt.imshow(prediction_results['visualizations'][0])
        plt.title(f"Prediction: {os.path.basename(image_path)}")
        plt.axis('off')
        plt.show()
else:
    logger.warning(f"Image not found: {image_path}")

## 7. Advanced Visualization & Analysis

In [ ]:
# Display confusion matrix
cm_path = os.path.join(config['output_dir'], "confusion_matrix_normalized.png")
if os.path.exists(cm_path):
    plt.figure(figsize=(8, 8))
    plt.imshow(Image.open(cm_path))
    plt.title("Normalized Confusion Matrix")
    plt.axis('off')
    plt.show()

# Display error analysis
error_analysis_dir = os.path.join(config['output_dir'], 'error_analysis')
if os.path.exists(error_analysis_dir):
    error_maps = glob.glob(os.path.join(error_analysis_dir, 'error_map_*.png'))
    if error_maps:
        plt.figure(figsize=(10, 10))
        plt.imshow(Image.open(error_maps[0]))
        plt.title(f"Error Analysis Map")
        plt.axis('off')
        plt.show()

## End of Pipeline

All artifacts are saved in the directory specified by `config['output_dir']`.